In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.medallion_data.ml_cache;

In [0]:
import os

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/medallion_data/ml_cache"


In [0]:
# ============================================================
# 0. IMPORTS
# MAGIC GAMMA TELESCOPE
# PySpark ML MLOps Notebook - Logistic Regression
# ============================================================

import mlflow
import mlflow.pyspark.ml

from pyspark.ml.connect.classification import LogisticRegression
from pyspark.ml.connect.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.sql import functions as F


In [0]:
# ============================================================
# 1. LOAD GOLD TABLE
# ============================================================

gold_table_name = "workspace.medallion_data.gold_telescope"

gold_df = spark.table(gold_table_name)

display(gold_df.limit(10))
gold_df.printSchema()

print("Gold table loaded:", gold_table_name)
print("Gold row count:", gold_df.count())
print("Gold column count:", len(gold_df.columns))

In [0]:
# ============================================================
# 2. CHECK LABEL DISTRIBUTION
# ============================================================

display(
    gold_df
    .groupBy("label")
    .count()
    .orderBy("label")
)

In [0]:
# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = gold_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

In [0]:
# ============================================================
# 4. DEFINE PYSPARK LOGISTIC REGRESSION MODEL
# ============================================================

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100
)

In [0]:
# ============================================================
# 5. TRAIN LOGISTIC REGRESSION WITH MLFLOW
# Serverless-safe version: NO autolog, NO model registration yet
# ============================================================

import mlflow
from pyspark.ml.classification import LogisticRegression as ClassicLogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator as ClassicMulticlassClassificationEvaluator, BinaryClassificationEvaluator as ClassicBinaryClassificationEvaluator

mlflow.set_experiment("/Users/maria.laramoran27@ncf.edu/gamma_telescope_pyspark_mlops")

with mlflow.start_run(run_name="Telescope_Production_Model_PySpark_LogisticRegression") as run:

    # Train model
    lr = ClassicLogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        maxIter=100
    )
    model = lr.fit(train_df)

    # Generate predictions
    predictions = model.transform(test_df)

    # Evaluators
    accuracy_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )

    f1_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1"
    )

    precision_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedPrecision"
    )

    recall_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedRecall"
    )

    auc_evaluator = ClassicBinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    # Metrics
    test_accuracy = accuracy_evaluator.evaluate(predictions)
    test_f1 = f1_evaluator.evaluate(predictions)
    test_precision = precision_evaluator.evaluate(predictions)
    test_recall = recall_evaluator.evaluate(predictions)
    test_auc = auc_evaluator.evaluate(predictions)

    # Log params manually
    mlflow.log_param("model_type", "PySpark LogisticRegression")
    mlflow.log_param("gold_table", gold_table_name)
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("regParam", 0.01)
    mlflow.log_param("elasticNetParam", 0.0)

    # Log metrics manually
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_precision_weighted", test_precision)
    mlflow.log_metric("test_recall_weighted", test_recall)
    mlflow.log_metric("test_f1_weighted", test_f1)
    mlflow.log_metric("test_auc", test_auc)

    print("Test Metrics:")
    print(f"test_accuracy: {test_accuracy:.4f}")
    print(f"test_precision_weighted: {test_precision:.4f}")
    print(f"test_recall_weighted: {test_recall:.4f}")
    print(f"test_f1_weighted: {test_f1:.4f}")
    print(f"test_auc: {test_auc:.4f}")

    run_id = run.info.run_id
    print("MLflow Run ID:", run_id)

In [0]:
# ============================================================
# 6. TRAIN & TUNE LOGISTIC REGRESSION WITH CROSS-VALIDATION & MLFLOW
# Serverless-safe version: NO autolog, NO model registration yet
# ============================================================

import mlflow
from pyspark.ml.classification import LogisticRegression as ClassicLogisticRegression
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator as ClassicMulticlassClassificationEvaluator,
    BinaryClassificationEvaluator as ClassicBinaryClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

mlflow.set_experiment("/Users/maria.laramoran27@ncf.edu/gamma_telescope_pyspark_mlops")

with mlflow.start_run(run_name="Telescope_Production_Model_PySpark_LR_CV") as run:

    # 1. Define the base estimator
    lr = ClassicLogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        maxIter=100
    )

    # 2. Build the parameter grid for hyperparameter tuning
    paramGrid = (
        ParamGridBuilder()
        .addGrid(lr.regParam, [0.01, 0.1, 1.0])
        .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
        .build()
    )

    # 3. Define the evaluator for Cross Validation
    cv_evaluator = ClassicBinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    # 4. Set up the CrossValidator
    cv = CrossValidator(
        estimator=lr,
        estimatorParamMaps=paramGrid,
        evaluator=cv_evaluator,
        numFolds=5,
        parallelism=2,
        seed=42
    )

    # 5. Fit CrossValidator to the training data
    print("Starting 5-fold Cross-Validation... this may take a moment.")
    cv_model = cv.fit(train_df)
    best_model = cv_model.bestModel

    # 6. Generate predictions for both training and holdout test sets
    train_predictions = best_model.transform(train_df)
    test_predictions = best_model.transform(test_df)

    # 7. Define evaluators for final metrics
    accuracy_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    )
    f1_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1"
    )
    precision_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
    )
    recall_evaluator = ClassicMulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedRecall"
    )
    auc_evaluator = ClassicBinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    )

    # 8. Calculate comparable training and test metrics
    train_accuracy = accuracy_evaluator.evaluate(train_predictions)
    train_f1 = f1_evaluator.evaluate(train_predictions)
    train_precision = precision_evaluator.evaluate(train_predictions)
    train_recall = recall_evaluator.evaluate(train_predictions)
    train_auc = auc_evaluator.evaluate(train_predictions)

    test_accuracy = accuracy_evaluator.evaluate(test_predictions)
    test_f1 = f1_evaluator.evaluate(test_predictions)
    test_precision = precision_evaluator.evaluate(test_predictions)
    test_recall = recall_evaluator.evaluate(test_predictions)
    test_auc = auc_evaluator.evaluate(test_predictions)

    # 9. Extract best hyperparameters and cross-validation score
    best_reg_param = best_model.getRegParam()
    best_elastic_net_param = best_model.getElasticNetParam()
    best_max_iter = best_model.getMaxIter()
    best_cv_auc = max(cv_model.avgMetrics)

    # 10. Log params manually to MLflow
    mlflow.log_param("model_type", "PySpark LogisticRegression CV")
    mlflow.log_param("gold_table", gold_table_name)
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("maxIter", best_max_iter)
    mlflow.log_param("best_regParam", best_reg_param)
    mlflow.log_param("best_elasticNetParam", best_elastic_net_param)

    # 11. Log metrics manually to MLflow
    mlflow.log_metric("cv_best_avg_auc", best_cv_auc)
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_precision_weighted", train_precision)
    mlflow.log_metric("train_recall_weighted", train_recall)
    mlflow.log_metric("train_f1_weighted", train_f1)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_precision_weighted", test_precision)
    mlflow.log_metric("test_recall_weighted", test_recall)
    mlflow.log_metric("test_f1_weighted", test_f1)
    mlflow.log_metric("test_auc", test_auc)

    print("\n--- Best Model Parameters ---")
    print(f"regParam: {best_reg_param}")
    print(f"elasticNetParam: {best_elastic_net_param}")
    print(f"best_cv_avg_auc: {best_cv_auc:.4f}")

    print("\n--- Training Metrics ---")
    print(f"train_accuracy: {train_accuracy:.4f}")
    print(f"train_precision_weighted: {train_precision:.4f}")
    print(f"train_recall_weighted: {train_recall:.4f}")
    print(f"train_f1_weighted: {train_f1:.4f}")
    print(f"train_auc: {train_auc:.4f}")

    print("\n--- Test Metrics ---")
    print(f"test_accuracy: {test_accuracy:.4f}")
    print(f"test_precision_weighted: {test_precision:.4f}")
    print(f"test_recall_weighted: {test_recall:.4f}")
    print(f"test_f1_weighted: {test_f1:.4f}")
    print(f"test_auc: {test_auc:.4f}")

    run_id = run.info.run_id
    print("\nMLflow Run ID:", run_id)